**V1 experiment.** This notebook contains the V1 implementation and experiment configuration. See [methods and evaluation](../../docs/METHODS.md) for the implemented protocol.


# Level 2 — ResNet-18 baseline with FULL training labels (50 epochs)

Companion to `Level2/level2_classification.ipynb`. The Level 2 ResNet-18
baseline used only 50% of the train-split clips (to match the graph SSL label
fraction) and ran for 15 epochs. Here we give the CNN every advantage:

- **100% of HF train-split labels** (no semi-supervised masking).
- **50 epochs** of Adam at lr=1e-3.
- Same model architecture (`utils.cnn.make_resnet18`) and same per-spectrogram
  normalization (`utils.cnn._normalize_specs`).

The aim is to see how a fully-supervised ResNet-18 compares against the
label-propagation graph SSL result of 84.7% accuracy (best params, k=3,
σ=self_tuning) from `level2_classification.ipynb`.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.cnn import make_resnet18, _normalize_specs
from utils.evaluation import plot_confusion_matrix

OUT_DIR = PROJECT_ROOT / "Level2" / "Resnet-18"
DATA_FILE = PROJECT_ROOT / "Level2" / "Data" / "level2_spectrograms.npz"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("project root:", PROJECT_ROOT)
print("data file:", DATA_FILE)
print("out dir:", OUT_DIR)


## Load Mel spectrograms and split by `is_train`


In [ ]:
data = np.load(DATA_FILE, allow_pickle=True)
S = data["S"].astype(np.float32)
y = data["labels"].astype(np.int64)
label_names = list(data["label_names"])
is_train = data["is_train"].astype(bool)

S_train = S[is_train]; y_train = y[is_train]
S_test = S[~is_train];  y_test = y[~is_train]

print(f"S full:   {S.shape}")
print(f"S train:  {S_train.shape}  ({is_train.sum()} windows from HF train clips)")
print(f"S test:   {S_test.shape}  ({(~is_train).sum()} windows from HF test clips)")
print(f"classes:  {len(label_names)} species")

print("\nTrain class distribution:")
for i, name in enumerate(label_names):
    print(f"  {name}: {(y_train == i).sum()}")
print("\nTest class distribution:")
for i, name in enumerate(label_names):
    print(f"  {name}: {(y_test == i).sum()}")


## Build tensors and dataloaders

Same per-spectrogram normalization as `utils.cnn.train_resnet_baseline` — the
Level 2 baseline used `_normalize_specs`, so reusing it keeps the comparison
apples-to-apples.


In [ ]:
EPOCHS = 50
BATCH_SIZE = 32
LR = 1e-3
SEED = 0

torch.manual_seed(SEED)
np.random.seed(SEED)

Xtr = _normalize_specs(S_train)[:, None, :, :]
Xte = _normalize_specs(S_test)[:, None, :, :]

Xtr_t = torch.from_numpy(Xtr)
ytr_t = torch.from_numpy(y_train)
Xte_t = torch.from_numpy(Xte)
yte_t = torch.from_numpy(y_test)

dl_tr = DataLoader(TensorDataset(Xtr_t, ytr_t),
                   batch_size=BATCH_SIZE, shuffle=True, drop_last=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print(f"epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LR}")


## Train ResNet-18 for 50 epochs

Hand-rolled training loop here (rather than calling `train_resnet_baseline`)
because we want per-epoch test accuracy in addition to train loss.


In [ ]:
model = make_resnet18(len(label_names)).to(device)
opt = optim.Adam(model.parameters(), lr=LR)
crit = nn.CrossEntropyLoss()

train_losses = []
train_accs = []
test_accs = []

def eval_acc(model, X_t, y_t):
    model.eval()
    with torch.no_grad():
        preds = []
        for i in range(0, X_t.shape[0], BATCH_SIZE):
            xb = X_t[i:i+BATCH_SIZE].to(device)
            preds.append(model(xb).argmax(dim=1).cpu().numpy())
    return float(accuracy_score(y_t.numpy(), np.concatenate(preds))) if preds else 0.0

for ep in range(EPOCHS):
    model.train()
    ep_loss = 0.0
    correct = 0
    total = 0
    for xb, yb in dl_tr:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        out = model(xb)
        loss = crit(out, yb)
        loss.backward()
        opt.step()
        ep_loss += loss.item() * xb.size(0)
        correct += (out.argmax(dim=1) == yb).sum().item()
        total += xb.size(0)
    train_losses.append(ep_loss / max(total, 1))
    train_accs.append(correct / max(total, 1))
    test_accs.append(eval_acc(model, Xte_t, yte_t))
    print(f"  epoch {ep+1:>2d}/{EPOCHS}: "
          f"train_loss={train_losses[-1]:.4f}  "
          f"train_acc={train_accs[-1]:.4f}  "
          f"test_acc={test_accs[-1]:.4f}")


## Final evaluation on the test set


In [ ]:
model.eval()
with torch.no_grad():
    preds = []
    for i in range(0, Xte_t.shape[0], BATCH_SIZE):
        xb = Xte_t[i:i+BATCH_SIZE].to(device)
        preds.append(model(xb).argmax(dim=1).cpu().numpy())
y_pred = np.concatenate(preds)

final_acc = accuracy_score(y_test, y_pred)
final_f1 = f1_score(y_test, y_pred, average="macro",
                    labels=range(len(label_names)), zero_division=0)
print(f"\nFinal test accuracy: {final_acc:.4f}")
print(f"Final macro F1:      {final_f1:.4f}")


## Plots — loss curve, accuracy curves, confusion matrix


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, EPOCHS + 1), train_losses, marker="o", markersize=3)
ax.set_xlabel("epoch")
ax.set_ylabel("train loss")
ax.set_title("Level 2 ResNet-18 (full labels, 50 epochs) — training loss")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "training_loss.png", dpi=150)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, EPOCHS + 1), train_accs, marker="o", markersize=3, label="train acc")
ax.plot(range(1, EPOCHS + 1), test_accs, marker="s", markersize=3, label="test acc")
ax.set_xlabel("epoch")
ax.set_ylabel("accuracy")
ax.set_title("Level 2 ResNet-18 (full labels, 50 epochs) — accuracy curves")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "accuracy_curve.png", dpi=150)
plt.close(fig)

plot_confusion_matrix(
    y_test, y_pred, label_names,
    save_path=OUT_DIR / "confusion_matrix.png",
    title="Level 2 ResNet-18 (full labels, 50 epochs) — test confusion matrix",
)
print("saved figures to", OUT_DIR)


## Comparison summary


In [ ]:
print("=== Level 2 — comparison ===")
print(f"  ResNet-18 (50% labels, 15 epochs):           77.9%   [from Level 2 results]")
print(f"  ResNet-18 (100% labels, 50 epochs):          {final_acc*100:.1f}%   [new result]")
print(f"  Label propagation (50% labels, default):     78.9%   [k=10, sigma=median]")
print(f"  Label propagation (50% labels, BEST params): 84.7%   [k=3, sigma=self_tuning]")
